# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Signal checks

**CTR vs position — CONFIRMED**

The data supports our hypothesis that CTR is related to search position and should therefore be considered relative to position.

**Engagement — FALSE**

Low engagement is too common in the observed data to be a useful standalone prioritization signal, because using it would flag a large proportion of the available observations.

### Baseline rule

Prioritize a content page for review when its CTR is at least **30% below the median CTR of its position peers**.

Position peers are grouped into **10-position intervals** (1–10, 11–20, 21–30, etc.). A position group must have at least **15 observations** before its median CTR is used as the peer benchmark.

The 30% gap is used as a starting baseline threshold to identify pages whose CTR is meaningfully below their position peers.

### Reason code

`CTR_BELOW_POSITION_PEER`

### Action label

`REVIEW_SNIPPET_HEADING`

The action is to review the page's search snippet and heading because the page is receiving lower CTR than its position peers.


CTR vs position Signal Check

Setting the treshold to 0.5%, 1% and 2% and checking the percentage of pages bellow these thresholds

In [6]:
# Check how the proportion of low-CTR observations changes across
# position groups at three different CTR thresholds: 0.5%, 1%, and 2%.

ctr_threshold_check = con.sql(
    f"""
    WITH base AS (
        SELECT
            gsc_avg_position,
            gsc_clicks,
            gsc_impressions,

            -- Calculate CTR as a percentage.
            CASE
                WHEN gsc_impressions > 0
                THEN (gsc_clicks * 100.0) / gsc_impressions
                ELSE NULL
            END AS ctr

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the same March 2026 window and require both
        -- GSC and GA4 data to be available.
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid search position.
          AND gsc_avg_position >= 1
    ),

    bucketed AS (
        SELECT
            -- Create the same 10-position groups used previously.
            FLOOR((gsc_avg_position - 1) / 10) * 10 + 1 AS position_start,
            ctr

        FROM base

        -- Only use observations where CTR can be calculated.
        WHERE ctr IS NOT NULL
    )

    SELECT
        -- Create a readable position-group label.
        CAST(position_start AS INTEGER) || '-' ||
        CAST(position_start + 9 AS INTEGER) AS position_bucket,

        -- Number of observations in each position group.
        COUNT(*) AS n,

        -- Percentage of observations below the 0.5% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 0.5) / COUNT(*),
            2
        ) AS below_0_5_pct,

        -- Percentage of observations below the 1% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 1.0) / COUNT(*),
            2
        ) AS below_1_pct,

        -- Percentage of observations below the 2% CTR threshold.
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ctr < 2.0) / COUNT(*),
            2
        ) AS below_2_pct

    FROM bucketed

    GROUP BY position_start

    -- Keep position groups in ascending order.
    ORDER BY position_start
    """
)

# Display the results for all three CTR thresholds.
ctr_threshold_check

┌─────────────────┬────────┬───────────────┬─────────────┬─────────────┐
│ position_bucket │   n    │ below_0_5_pct │ below_1_pct │ below_2_pct │
│     varchar     │ int64  │    double     │   double    │   double    │
├─────────────────┼────────┼───────────────┼─────────────┼─────────────┤
│ 1-10            │ 188953 │         52.94 │       68.12 │       83.07 │
│ 11-20           │  72442 │         64.11 │       76.82 │       88.83 │
│ 21-30           │  51657 │         77.26 │       87.51 │       94.52 │
│ 31-40           │  27970 │          86.8 │       93.07 │       96.49 │
│ 41-50           │   9565 │         91.23 │       94.46 │        96.4 │
│ 51-60           │   2547 │         91.79 │       93.76 │       95.05 │
│ 61-70           │   1076 │         92.75 │        93.4 │       94.33 │
│ 71-80           │    602 │         94.02 │       94.85 │       95.51 │
│ 81-90           │    356 │         94.66 │       95.22 │       96.35 │
│ 91-100          │    190 │         95.79 │       

Engagement Signal Check

In [8]:
# Check the distribution of engagement rates in 10-percentage-point buckets.
# Engagement rate is defined as engaged sessions divided by all sessions.

engagement_bucket_check = con.sql(
    f"""
    WITH base AS (
        SELECT
            ga4_sessions,
            ga4_engaged_sessions,

            -- Calculate engagement rate as engaged sessions divided
            -- by all sessions, expressed as a percentage.
            CASE
                WHEN ga4_sessions > 0
                THEN (ga4_engaged_sessions * 100.0) / ga4_sessions
                ELSE NULL
            END AS engagement_rate

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the same March 2026 window as the CTR check.
        -- Both GSC and GA4 data must be available.
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ),

    bucketed AS (
        SELECT
            engagement_rate,

            -- Create 10-percentage-point buckets:
            -- 0-10%, 10-20%, 20-30%, and so on.
            FLOOR(engagement_rate / 10) * 10 AS bucket_start

        FROM base

        -- Exclude observations where engagement rate cannot be calculated.
        WHERE engagement_rate IS NOT NULL
          AND engagement_rate >= 0
          AND engagement_rate <= 100
    )

    SELECT
        -- Create a readable engagement bucket label.
        CAST(bucket_start AS INTEGER) || '-' ||
        CAST(bucket_start + 10 AS INTEGER) || '%' AS engagement_bucket,

        -- n = number of observations in each engagement bucket.
        COUNT(*) AS n

    FROM bucketed

    GROUP BY bucket_start

    -- Display the buckets from lowest to highest engagement.
    ORDER BY bucket_start
    """
)

# Display the corrected engagement bucket table.
engagement_bucket_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────┐
│ engagement_bucket │   n    │
│      varchar      │ int64  │
├───────────────────┼────────┤
│ 0-10%             │ 338188 │
│ 10-20%            │   3207 │
│ 20-30%            │   3364 │
│ 30-40%            │   2863 │
│ 40-50%            │    145 │
│ 50-60%            │   4858 │
│ 60-70%            │    214 │
│ 70-80%            │     13 │
│ 100-110%          │   8243 │
└───────────────────┴────────┘

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Aggregate to page level

In [13]:
# Aggregate the daily performance data to one row per content page and client.
# This prevents the same page from appearing once for every day in the queue.

page_level = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Add all impressions across the March 2026 window.
        SUM(gsc_impressions) AS total_impressions,

        -- Add all clicks across the March 2026 window.
        SUM(gsc_clicks) AS total_clicks,

        -- Use the median daily search position as the page's
        -- representative position for the month.


        ROUND(
            MEDIAN(
              CASE
                  WHEN gsc_avg_position >= 1
                  THEN gsc_avg_position
                  ELSE NULL
              END
           ), 2
      ) AS median_position

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use the same March 2026 analysis window.
    WHERE month = '2026-03'

      -- Require both data sources to be available.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
)

# Display a small sample so we can inspect the page-level result.
page_level.limit(10)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │
│         varchar         │         varchar          │      int128       │    int128    │     double      │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │
│ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │              3943 │           23 │             4.1 │
│ client_65de48885f4ef01b │ content_3c286ded8bd68120 │              2180 │           15 │            8.37 │
│ client_65de48885f4ef01b │ content_b2108e8fe3360fa6 │               503 │            8 │            5.15 │
│ client_65de48885f4ef01b │ content_ff867882e604fa96 │                24 │            0 │            2.85 │
│ client_65de48885f4ef01b │ 

In [14]:
# Calculate each page's monthly CTR and assign it to a 10-position
# search-ranking bucket.

page_features = con.sql(
    """
    SELECT
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        median_position,

        -- Calculate the page's CTR as a percentage.
        CASE
            WHEN total_impressions > 0
            THEN (total_clicks * 100.0) / total_impressions
            ELSE NULL
        END AS page_ctr,

        -- Group pages into 10-position intervals:
        -- 1-10, 11-20, 21-30, and so on.
        FLOOR((median_position - 1) / 10) * 10 + 1
            AS position_bucket_start

    FROM page_level

    -- Only retain pages with a valid search position.
    WHERE median_position >= 1

      -- Only retain pages where CTR can be calculated.
      AND total_impressions > 0
    """
)

# Display a sample of the page-level features.
page_features.limit(10)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬─────────────────────┬───────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │      page_ctr       │ position_bucket_start │
│         varchar         │         varchar          │      int128       │    int128    │     double      │       double        │        double         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼─────────────────────┼───────────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │  0.4366812227074236 │                   1.0 │
│ client_65de48885f4ef01b │ content_e25ea7297a1dffd3 │              3943 │           23 │             4.1 │  0.5833121988333756 │                   1.0 │
│ client_65de48885f4ef01b │ content_3c286ded8bd68120 │              2180 │  

In [15]:
# Calculate the peer-group size and median CTR for each search-position bucket.
# These values will become the benchmark used to score each page.

peer_benchmarks = con.sql(
    """
    SELECT
        position_bucket_start,

        -- Count the number of pages in each position group.
        COUNT(*) AS peer_n,

        -- Calculate the median page CTR within each position group.
        MEDIAN(page_ctr) AS peer_median_ctr

    FROM page_features

    GROUP BY position_bucket_start

    -- Show the position groups in ranking order.
    ORDER BY position_bucket_start
    """
)

# Display the peer-group benchmarks so we can inspect them.
peer_benchmarks

┌───────────────────────┬────────┬─────────────────────┐
│ position_bucket_start │ peer_n │   peer_median_ctr   │
│        double         │ int64  │       double        │
├───────────────────────┼────────┼─────────────────────┤
│                   1.0 │  37686 │  0.5813953488372093 │
│                  11.0 │  10765 │ 0.33076074972436603 │
│                  21.0 │   6636 │   0.120796326559979 │
│                  31.0 │   3662 │                 0.0 │
│                  41.0 │   1603 │                 0.0 │
│                  51.0 │    682 │                 0.0 │
│                  61.0 │    426 │                 0.0 │
│                  71.0 │    248 │                 0.0 │
│                  81.0 │    143 │                 0.0 │
│                  91.0 │     83 │                 0.0 │
│                 101.0 │      5 │                 0.0 │
│                 111.0 │      3 │                 0.0 │
│                 121.0 │      2 │                50.0 │
│                 131.0 │      

In [18]:
# Recalculate the baseline score and cap negative values at zero.
# A negative raw score means the page CTR is above its peer median,
# so it receives a score of zero because our rule focuses on underperformance.

scored_pages = con.sql(
    """
    SELECT
        p.client_hash_id,
        p.content_hash_id,
        p.total_impressions,
        p.total_clicks,
        p.median_position,
        p.page_ctr,
        p.position_bucket_start,

        -- Add the number of observations in the position peer group.
        b.peer_n,

        -- Add the median CTR of the position peer group.
        b.peer_median_ctr,

        -- Calculate the relative CTR underperformance.
        -- MAX(0, ...) prevents pages performing above their peers
        -- from receiving negative scores.
        GREATEST(
            0,
            1 - (p.page_ctr / b.peer_median_ctr)
        ) AS baseline_score

    FROM page_features AS p

    INNER JOIN peer_benchmarks AS b
        ON p.position_bucket_start = b.position_bucket_start

    -- Require at least 10 observations in the peer group.
    WHERE b.peer_n >= 10

      -- Exclude groups where the peer median CTR is zero.
      AND b.peer_median_ctr > 0

      -- Only retain pages where CTR can be calculated.
      AND p.page_ctr IS NOT NULL
    """
)

# Display a sample of the scores.
scored_pages.limit(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬─────────────────────┬───────────────────────┬────────┬─────────────────────┬─────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │      page_ctr       │ position_bucket_start │ peer_n │   peer_median_ctr   │   baseline_score    │
│         varchar         │         varchar          │      int128       │    int128    │     double      │       double        │        double         │ int64  │       double        │       double        │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼─────────────────────┼───────────────────────┼────────┼─────────────────────┼─────────────────────┤
│ client_65de48885f4ef01b │ content_b1f61fc81b28b2d4 │               458 │            2 │            4.39 │  0.4366812227074236 │                   1.0 │  37686 │  0.581395

In [27]:
# Build the complete ranked baseline queue.
# Every eligible page is retained and ranked by its baseline score.
# The 30% threshold determines whether the page receives an action.

baseline_queue = con.sql(
    """
    SELECT
        client_hash_id,
        content_hash_id,
        total_impressions,
        total_clicks,
        median_position,
        page_ctr,
        baseline_score,

        -- Assign the single reason code when the rule fires.
        CASE
            WHEN baseline_score >= 0.30
            THEN 'CTR_BELOW_POSITION_PEER'
            ELSE NULL
        END AS reason_code,

        -- Assign the action associated with the reason code.
        CASE
            WHEN baseline_score >= 0.30
            THEN 'REVIEW_SNIPPET_HEADING'
            ELSE 'NO_ACTION'
        END AS action_label

    FROM scored_pages

    -- Rank all eligible pages from highest to lowest score.
    ORDER BY
        baseline_score DESC,
        total_impressions DESC,
        content_hash_id
    """
)

# Display the first 20 rows of the final queue.
baseline_queue.limit(2000)

┌─────────────────────────┬──────────────────────────┬───────────────────┬──────────────┬─────────────────┬──────────┬────────────────┬─────────────────────────┬────────────────────────┐
│     client_hash_id      │     content_hash_id      │ total_impressions │ total_clicks │ median_position │ page_ctr │ baseline_score │       reason_code       │      action_label      │
│         varchar         │         varchar          │      int128       │    int128    │     double      │  double  │     double     │         varchar         │        varchar         │
├─────────────────────────┼──────────────────────────┼───────────────────┼──────────────┼─────────────────┼──────────┼────────────────┼─────────────────────────┼────────────────────────┤
│ client_23a62021009f63c4 │ content_bf078007df823490 │             37262 │            0 │            7.96 │      0.0 │            1.0 │ CTR_BELOW_POSITION_PEER │ REVIEW_SNIPPET_HEADING │
│ client_23a62021009f63c4 │ content_bd63db2d0757e760 │           

In [ ]:
# Write the final ranked baseline queue to the required CSV path.

baseline_queue.write_csv(
    "work/outputs/baseline_action_score.csv",
    overwrite=True
)

print("Baseline queue written successfully.")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.